In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from scipy.stats import norm

from snail.damage_library import get_metadata, load_curve
from snail.damages import calculate_direct_damages
from snail.overlay import overlay_rasters

# Support running Jupyter from either the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent

# from St. Lucia project
road_path = repo_root / "data/lca/asset/Roads_rmms_2018.gpkg"
flood_dir = repo_root / "data/lca/hazard/flood"
flood_return_periods = (20, 50, 100, 200, 500, 1500)
flood_paths = {
    f"flood_rp{rp}": flood_dir / f"LC_2016_FLRF_U_RP{rp}_RD_30m_4326.tif"
    for rp in flood_return_periods
}
assert road_path.exists() and all(path.exists() for path in flood_paths.values())

flood_rasters = pd.DataFrame(
    {"path": list(flood_paths.values()), "key": list(flood_paths.keys())}
)

In [ ]:
roads = gpd.read_file(road_path, layer="Roads_RMMS_2018")
curve_metadata = {curve_id: get_metadata(curve_id) for curve_id in ("F7.2", "F7.3")}
curves = {curve_id: load_curve(curve_id) for curve_id in curve_metadata}

print(f"Road features: {len(roads):,}")
print(f"Road CRS: {roads.crs}")
for curve_id, metadata in curve_metadata.items():
    print(
        f"{curve_id}: {metadata.hazard_name} hazard, "
        f"{metadata.exposed_element}, intensity unit {metadata.intensity_unit}"
    )
roads.head()

In [ ]:
splits = overlay_rasters(roads, flood_rasters)

# Recompute lengths on the shared split geometries in metres, then convert to km.
splits["length_km"] = (
    splits.geometry.to_crs("EPSG:32620").length / 1_000
)
valid_depth = np.ones(len(splits), dtype=bool)
for case, path in flood_paths.items():
    with rasterio.open(path) as flood_raster:
        flood_nodata = flood_raster.nodata
    depth = splits[case]
    case_valid = np.isfinite(depth) & (depth >= 0)
    if flood_nodata is not None:
        case_valid &= depth != flood_nodata
    valid_depth &= case_valid

damage_splits = splits.loc[valid_depth].copy()
print(f"Split segments: {len(splits):,}")
print(f"Segments with valid depth in every return-period map: {len(damage_splits):,}")
print(f"Road assets represented: {damage_splits['ogc_fid'].nunique():,}")
damage_splits[["ogc_fid", "length_km", *flood_paths]].head()

In [ ]:
damage_summary = calculate_direct_damages(
    damage_splits,
    asset_id="ogc_fid",
    length="length_km",
    intensity_columns={"flood_rp200": "flood_rp200"},
    curves=curves,
    curve_weights={"F7.2": 0.3, "F7.3": 0.7},
    cost_mean=300_000,
    cost_system_sd=75_000,
    hazard_alpha={"flood_rp200": 0.20},
    eta_sd=0.1,
    n_samples=1_000,
    quantiles=(0.05, 0.25, 0.5, 0.75, 0.95),
    seed=42,
)

damage_summary.quantiles.head()

In [ ]:
depth = np.linspace(0, 4, 200)
eta_sd = 0.1
eta_05, eta_95 = norm.ppf(0.05) * eta_sd, norm.ppf(0.95) * eta_sd

fig, ax = plt.subplots(figsize=(7, 4))
for curve_id, curve in curves.items():
    central_damage = curve.damage_fraction(depth)
    damage_05 = np.clip(central_damage * np.exp(eta_05 - 0.5 * eta_sd**2), 0, 1)
    damage_95 = np.clip(central_damage * np.exp(eta_95 - 0.5 * eta_sd**2), 0, 1)
    (line,) = ax.plot(depth, central_damage, label=f"{curve_id}")
    ax.fill_between(
        depth, damage_05, damage_95, color=line.get_color(), alpha=0.15,
        label=f"{curve_id} p5–95",
    )
ax.set(xlabel="Flood depth (m)", ylabel="Damage fraction", ylim=(0, 1), title="Road flood damage curves")
ax.legend()
fig.tight_layout()

In [ ]:
asset_results = damage_summary.quantiles.sort_values("q_50").tail(20).copy()
asset_results["asset_label"] = asset_results["ogc_fid"].astype(str)

box_stats = [
    {
        "label": row.asset_label,
        "whislo": row.q_05,
        "q1": row.q_25,
        "med": row.q_50,
        "q3": row.q_75,
        "whishi": row.q_95,
        "fliers": [],
    }
    for row in asset_results.itertuples()
]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bxp(box_stats, vert=False, showfliers=False, patch_artist=True,
       boxprops={"facecolor": "tab:orange", "alpha": 0.55},
       medianprops={"color": "black"}, whiskerprops={"color": "0.4"},
       capprops={"color": "0.4"})
ax.set(xlabel="Direct damage (currency units)", ylabel="Road asset ID (ogc_fid)", title="Sampled direct damage by road asset")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()

In [ ]:
fine_quantiles = np.arange(0.01, 1.00, 0.01)
fine_damage_summary = calculate_direct_damages(
    damage_splits,
    asset_id="ogc_fid",
    length="length_km",
    intensity_columns={"flood_rp200": "flood_rp200"},
    curves=curves,
    curve_weights={"F7.2": 0.3, "F7.3": 0.7},
    cost_mean=300_000,
    cost_system_sd=75_000,
    hazard_alpha={"flood_rp200": 0.1},
    eta_sd=0.1,
    n_samples=1000,
    quantiles=fine_quantiles,
    seed=42,
)

In [ ]:
# Shared systemic draws still leave local hazard and damage noise, so summing
# matching asset percentiles only approximates the total distribution.
# calculate_direct_damages discards the samples needed for exact aggregation.
quantile_columns = [f"q_{round(q * 100):02d}" for q in fine_quantiles]
total_quantiles = fine_damage_summary.quantiles[quantile_columns].sum(axis=0)

total_damage = total_quantiles.to_numpy()
total_pdf = np.gradient(fine_quantiles, total_damage)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(total_damage, total_pdf, color="tab:purple", s=2)
ax.fill_between(total_damage, total_pdf, color="tab:purple", alpha=0.2)
ax.set(
    xlabel="Total direct damage (illustrative currency units)",
    ylabel="Probability density (per currency unit)",
    title="Approximate total direct damage PDF",
)
ax.grid(alpha=0.25)
fig.tight_layout()

In [ ]:
ead_summary = calculate_direct_damages(
    damage_splits,
    asset_id="ogc_fid",
    length="length_km",
    intensity_columns={f"flood_rp{rp}": f"flood_rp{rp}" for rp in flood_return_periods},
    return_periods={f"flood_rp{rp}": rp for rp in flood_return_periods},
    curves=curves,
    curve_weights={"F7.2": 0.3, "F7.3": 0.7},
    cost_mean=300_000,
    cost_system_sd=75_000,
    hazard_alpha={f"flood_rp{rp}": 0.1 for rp in flood_return_periods},
    eta_sd=0.1,
    n_samples=1_000,
    quantiles=None,
    ead_quantiles=(0.05, 0.25, 0.5, 0.75, 0.95),
    seed=42,
)

ead_summary.ead_quantiles.head()

In [ ]:
ead_assets = ead_summary.ead_quantiles.sort_values("q_50").tail(20).copy()
ead_assets["asset_label"] = ead_assets["ogc_fid"].astype(str)

ead_box_stats = [
    {
        "label": row.asset_label,
        "whislo": row.q_05,
        "q1": row.q_25,
        "med": row.q_50,
        "q3": row.q_75,
        "whishi": row.q_95,
        "fliers": [],
    }
    for row in ead_assets.itertuples()
]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bxp(
    ead_box_stats,
    vert=False,
    showfliers=False,
    patch_artist=True,
    boxprops={"facecolor": "tab:purple", "alpha": 0.55},
    medianprops={"color": "black"},
    whiskerprops={"color": "0.4"},
    capprops={"color": "0.4"},
)
ax.set(
    xlabel="Annual expected direct damage (currency units)",
    ylabel="Road asset ID (ogc_fid)",
    title="Sampled EAD by road asset",
)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()

In [ ]:
result = calculate_direct_damages(
    damage_splits,
    asset_id="ogc_fid",
    length="length_km",
    intensity_columns={f"flood_rp{rp}": f"flood_rp{rp}" for rp in flood_return_periods},
    return_periods={f"flood_rp{rp}": rp for rp in flood_return_periods},
    curves=curves,
    curve_weights={"F7.2": 0.3, "F7.3": 0.7},
    cost_mean=300_000,
    cost_system_sd=75_000,
    hazard_alpha={f"flood_rp{rp}": 0.1 for rp in flood_return_periods},
    eta_sd=0.1,
    n_samples=1_000,
    quantiles=(0.05, 0.25, 0.5, 0.75, 0.95),
    ead_quantiles=(0.05, 0.25, 0.5, 0.75, 0.95),
    seed=42,
)

In [ ]:
result.quantiles